In [ ]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain.agents import tool
from typing import TypedDict, Annotated
import operator

### Defining Tools

In [3]:
@tool
def calculate_si(p, r, t):
    """This function calculates the simple interest given the principal amount, rate of interest and time period."""
    return (p * r * t) / 100

@tool
def calculate_ci(p, r, t):
    """This function calculates the compound interest given the principal amount, rate of interest and time period."""
    return p * (1 + r / 100) ** t - p

In [ ]:
print(type(calculate_si))

### Defining Agent State

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

### Defining Agent

In [6]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        ## creating graph
        graph = StateGraph(AgentState)
        ## adding nodes
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        ## adding conditional edges
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        ## adding edges
        graph.add_edge("action", "llm")
        ## setting entry point
        graph.set_entry_point("llm")
        ## compiling graph
        self.graph = graph.compile()
        ## binding tools to the model
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        print(f"Checking for action: {result}")
        return len(result.tool_calls) > 0

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
    model=model_name,
    temperature=0.0,
    max_tokens=1024
)

llm

In [8]:
prompt = """You are a smart accountant. Use the calculate simple interest and compound interest tools to calculate SI and CI. \
You are allowed to make multiple calls (either together or in sequence). \
"""

cbot = Agent(llm, [calculate_si, calculate_ci], system=prompt)

In [ ]:
messages = [HumanMessage(content="What is the simple interest on a loan of 1000 USD at 5% for 10 years?")]
result = cbot.graph.invoke({"messages": messages})

In [ ]:
messages = [HumanMessage(content="What is the difference between simple interest and compound interest on a loan of 1000 USD at 5% for 10 years?")]
result = cbot.graph.invoke({"messages": messages})
print(result['messages'][-1].content)